In [10]:
import pandas as pd
import numpy as np

def load_and_clean_data(country="Germany"):
    print(f"--- Finalizing Data Pipeline: {country} ---")
    
    # The function loads the primary datasets including prices, weather, load actuals, and forecasts.
    df_prices = pd.read_csv(f'../data/raw/{country}_Prices.csv')
    df_weather = pd.read_csv(f'../data/raw/{country}_weather.csv')
    df_act_load = pd.read_csv(f'../data/raw/{country}_Actual_Load.csv')
    df_for_load = pd.read_csv(f'../data/raw/{country}_Forecast_Load.csv')
    df_gen = pd.read_csv(f'../data/raw/{country}_Actual_Generation.csv')

    # The script standardizes column headers by removing whitespace and applying underscores.
    dataframes = [df_prices, df_weather, df_act_load, df_for_load, df_gen]
    for df in dataframes:
        df.columns = df.columns.str.strip().str.replace(' ', '_')

    # The logic renames value columns to differentiate between actual and forecasted demand.
    # This prevention of generic suffixes ensures each feature maintains a unique, descriptive identity.
    df_act_load.rename(columns={'Value_MW': 'Actual_Load_MW', 'Actual_Total_Load': 'Actual_Load_MW'}, inplace=True, errors='ignore')
    df_for_load.rename(columns={'Value_MW': 'Forecast_Load_MW', 'Forecasted_Load': 'Forecast_Load_MW'}, inplace=True, errors='ignore')
    
    # The system standardizes the price target variable.
    df_prices.rename(columns={'Price_EUR_MWh': 'Price', 'Prices': 'Price'}, inplace=True, errors='ignore')

    # The process aligns all timestamps to UTC and enforces a common join key.
    for df in dataframes:
        time_col = 'Timestamp' if 'Timestamp' in df.columns else df.columns[0]
        df[time_col] = pd.to_datetime(df[time_col], utc=True)
        df.rename(columns={time_col: 'Timestamp'}, inplace=True)
        df.ffill(inplace=True)

    # The function executes sequential inner merges to assemble the master dataset.
    df_gold = pd.merge(df_prices, df_weather, on='Timestamp', how='inner')
    df_gold = pd.merge(df_gold, df_act_load, on='Timestamp', how='inner')
    df_gold = pd.merge(df_gold, df_for_load, on='Timestamp', how='inner')
    df_gold = pd.merge(df_gold, df_gen, on='Timestamp', how='inner')

    # The process defines the temporal index and removes any duplicate hour entries.
    df_gold.set_index('Timestamp', inplace=True)
    df_gold.sort_index(inplace=True)
    df_gold = df_gold[~df_gold.index.duplicated(keep='first')]

    print(f"Final Gold Dataset Polished. Shape: {df_gold.shape}")
    return df_gold

# The main execution block initiates the pipeline.
df_gold = load_and_clean_data("Germany")

--- Finalizing Data Pipeline: Germany ---
Final Gold Dataset Polished. Shape: (2183, 15)


In [11]:
# Feature Engineering: Time, Behavior, and Memory

# The process extracts temporal and behavioral features from the timestamp index.
df_gold['Hour'] = df_gold.index.hour
df_gold['DayOfWeek'] = df_gold.index.dayofweek
df_gold['Is_Weekend'] = df_gold['DayOfWeek'].isin([5, 6]).astype(int)

# The system calculates a heating load proxy using the apparent temperature to estimate human heating demand.
df_gold['Heating_Demand_Proxy'] = np.maximum(0, 18 - df_gold['Apparent_Temp'])

# The logic computes the Load Forecast Error by comparing actual demand against forecasted values.
# Positive values indicate demand was higher than initially expected by the grid operator.
if 'Actual_Load_MW' in df_gold.columns and 'Forecast_Load_MW' in df_gold.columns:
    df_gold['Load_Forecast_Error'] = df_gold['Actual_Load_MW'] - df_gold['Forecast_Load_MW']

# The script defines the target variable to streamline the generation of market memory features.
target_col = 'Price'

# The code computes a 24-hour lag to capture the price at the exact hour on the previous day.
df_gold['Price_Lag_24h'] = df_gold[target_col].shift(24)

# The application generates a 168-hour lag to reflect the price at the exact hour during the previous week.
df_gold['Price_Lag_168h'] = df_gold[target_col].shift(168)

# The logic calculates a 24-hour rolling average to represent the general market trend over the preceding day.
df_gold['Price_Rolling_24h_Avg'] = df_gold[target_col].rolling(window=24).mean()

# The pipeline drops empty rows created by the shifting operations to ensure compatibility with machine learning models.
df_gold.dropna(inplace=True)

print(f"Gold Layer Feature Engineering Complete.")
print(f"Final Shape: {df_gold.shape}")
print(f"Total Features Ready for ML: {len(df_gold.columns)}")

Gold Layer Feature Engineering Complete.
Final Shape: (2015, 23)
Total Features Ready for ML: 23


In [12]:
# The system outputs an enumerated list of all final features engineered for the machine learning model.
print("Final Gold Layer Features:")
for index, feature in enumerate(df_gold.columns, start=1):
    print(f"{index}. {feature}")

Final Gold Layer Features:
1. Price
2. Temperature_2m
3. Apparent_Temp
4. Rel_Humidity_2m
5. Precipitation
6. Snowfall
7. Cloud_Cover
8. Shortwave_Rad
9. WindSpeed_10m
10. WindSpeed_100m
11. WindGusts
12. Actual_Load_MW
13. Forecast_Load_MW
14. Gen_MW
15. FuelType
16. Hour
17. DayOfWeek
18. Is_Weekend
19. Heating_Demand_Proxy
20. Load_Forecast_Error
21. Price_Lag_24h
22. Price_Lag_168h
23. Price_Rolling_24h_Avg


# Gold Layer: Feature Engineering & Data Synthesis

This notebook executes the transition from raw architectural data to a high-fidelity **Gold Layer** feature store. The primary objective is to synthesize physical grid constraints, meteorological variables, and market expectations into a unified, time-series dataset optimized for supervised machine learning and regression modeling.

## 1. Multi-Dimensional Data Integration
The pipeline performs a **6-way inner merge** to construct a holistic view of the European energy market. By aligning disparate sources on a unified UTC timestamp, the system integrates the following pillars:
* **Market Pricing:** Day-Ahead clearing prices (The Target Variable).
* **Meteorological Constraints:** 10 distinct weather variables including wind speed at multiple altitudes and solar radiation.
* **Physical Demand:** Actual total load (realized consumption).
* **Market Expectations:** Forecasted total load (day-ahead grid projections).
* **Generation Mix:** Actual generation by fuel type.
* **Supply Forecasts:** Day-ahead wind and solar production estimates.

---

## 2. Feature Engineering & Domain Logic
The notebook transforms raw values into predictive signals through specialized engineering layers:

* **Temporal & Behavioral Mapping:** Extraction of cyclical features such as **Hour**, **DayOfWeek**, and **Is_Weekend** to capture industrial and residential consumption patterns.
* **Domain Proxies:** Calculation of the **Heating_Demand_Proxy** using apparent temperature to quantify the non-linear relationship between thermal comfort and electrical load.
* **Market "Surprise" Metrics:** Generation of the **Load_Forecast_Error**. By calculating the delta between actual and forecasted load, the model can identify periods of unexpected scarcity or surplus that drive price volatility.

---

## 3. Time-Series Memory (Lags & Momentum)
To account for the high degree of autocorrelation in energy markets, the system implements a "Market Memory" layer:
* **Cyclical Lags:** **24-hour** and **168-hour (7-day)** price shifts allow the model to learn from the previous day's trends and the previous week's cycles.
* **Momentum Indicators:** **24-hour rolling averages** provide a smoothed baseline of the current market trajectory, filtering out hourly noise.

---

## 4. Data Quality & ML Readiness
The final stage of the notebook ensures the dataset is mathematically compatible with regression algorithms. This involves:
* **Suffix Resolution:** Explicitly renaming merged columns to avoid ambiguous identifiers (e.g., `_x`, `_y`) and ensuring unique naming for actuals vs. forecasts.
* **Timestamp Alignment:** Forcing a strict UTC timeline to resolve cross-border timezone conflicts and API inconsistencies.
* **Null-Value Management:** Strategic dropping of the initial 168 rows (7 days) to account for the "warm-up" period required by the 7-day lag features, resulting in a perfectly clean, gap-free training set.
